In [8]:
from langgraph.graph import START, END, StateGraph
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

In [9]:
load_dotenv()
llm = ChatCohere(model="command-r7b-12-2024")

In [10]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

def chat_node(state : ChatState):
    question = state['messages']

    answer = llm.invoke(question).content

    return {'messages': [answer]}

In [11]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

workflow = graph.compile(checkpointer=checkpointer)

In [14]:
thread_id = 1
while True:
    user = input("Enter ur msg:- ")
    print(user)
    if user.strip().lower() in ['bye','exit','stop']:
        break
    else:
        config = {'configurable':{'thread_id':thread_id}}
        res = workflow.invoke({
            'messages':[
                HumanMessage(content=user)
            ]
        }, config=config)
        print(res['messages'][-1].content)

Hi, my name is tanisha
Hello Tanisha! How can I help you today?
what is my name
Your name is Tanisha. Is there anything else I can help you with today?
can you add 10 and 1100
Sure! 10 + 1100 = 1110.
can u mulitply the result with 2
Sure! 1110 * 2 = 2220.
bye
